# Solutions · 1. Least squares with exact constraints: the KKT system

Worked solutions to the three exercises of [Lecture 1](../../notebooks/part1_conservation/01_kkt_equality_constrained_regression.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Unconstrained OLS vs. Exact KKT Conservation on a Methanol Reactor")
rng = np.random.default_rng(42)
N, d = 35, 5
X_scan = np.eye(d)[np.arange(N) % d] + rng.normal(0.0, 0.15, size=(N, d))
# True methanol reactor yields conserving 1 mol C, 6 mol H, 2 mol O:
w_true = np.array([0.24, 0.08, 0.32, 0.68, 2.20])
C_atom = np.array([
    [1.0, 1.0, 0.0, 1.0, 0.0],
    [4.0, 0.0, 2.0, 0.0, 2.0],
    [1.0, 1.0, 1.0, 2.0, 0.0],
])
d_atom = np.array([1.0, 6.0, 2.0])
y_obs = X_scan @ w_true + rng.normal(0.0, 0.14, size=N)

lam = 1e-8
w_ols = linear_physics.ridge_regression(X_scan, y_obs, lam=lam, penalize_intercept=True)
kkt_res = linear_physics.equality_constrained_lstsq(X_scan, y_obs, C_atom, d_atom, lam=lam)
w_kkt = kkt_res["w"]

print(f"{'Species':>8s} | {'True Yield':>10s} | {'Unconstrained OLS':>17s} | {'KKT Constrained':>15s}")
print("-" * 58)
for sp, wt, wo, wk in zip(["CH3OH", "CO", "H2O", "CO2", "H2"], w_true, w_ols, w_kkt):
    print(f"{sp:>8s} | {wt:10.4f} | {wo:17.4f} | {wk:15.4f}")
print("OLS elemental violation ||C w_OLS - d||_max :", float(np.max(np.abs(C_atom @ w_ols - d_atom))))
print("KKT elemental residual  ||C w_KKT - d||_max :", kkt_res["constraint_residual"])
assert kkt_res["constraint_residual"] < 1e-13
heading("Demonstration 2: Schur-Complement Oblique Projection & Dual Multipliers")
G = X_scan.T @ X_scan + lam * np.eye(d)
G_inv_CT = np.linalg.solve(G, C_atom.T)
schur = C_atom @ G_inv_CT
mu_schur = np.linalg.solve(schur, C_atom @ w_ols - d_atom)
w_schur = w_ols - G_inv_CT @ mu_schur

print("Schur complement matrix S = C G^(-1) C^T:\n", np.round(schur, 5))
print("Lagrange multipliers mu (KKT block)     :", np.round(kkt_res["lagrange_multipliers"], 6))
print("Lagrange multipliers mu (Schur formula) :", np.round(mu_schur, 6))
assert np.allclose(w_kkt, w_schur, atol=1e-13)
assert np.allclose(kkt_res["lagrange_multipliers"], mu_schur, atol=1e-13)
heading("Demonstration 3: Analytical Covariance Trace & Parameter Error Reduction")
err_ols = float(np.linalg.norm(w_ols - w_true))
err_kkt = float(np.linalg.norm(w_kkt - w_true))
G_inv = np.linalg.inv(G)
P_cov_kkt = G_inv - G_inv_CT @ np.linalg.solve(schur, G_inv_CT.T)
trace_ols = float(np.trace(G_inv))
trace_kkt = float(np.trace(P_cov_kkt))

print(f"Empirical L2 parameter error: OLS = {err_ols:.4f} -> KKT = {err_kkt:.4f}")
print(f"Normalized variance trace   : Tr(G^-1) = {trace_ols:.4f} -> Tr(Cov_KKT) = {trace_kkt:.4f} ({100*(1-trace_kkt/trace_ols):.1f}% reduction)")
assert err_kkt < err_ols and trace_kkt < 0.55 * trace_ols
heading("Demonstration 4: The Soft-Penalty Ill-Conditioning vs. Exact KKT")
print(f"{'Penalty mu_pen':>14s} | {'cond(H_soft)':>14s} | {'Balance Violation':>18s} | {'||w_soft - w_KKT||':>18s}")
print("-" * 72)
for mu_pen in [1e0, 1e2, 1e6, 1e10, 1e12]:
    H_soft = G + mu_pen * (C_atom.T @ C_atom)
    w_soft = np.linalg.solve(H_soft, X_scan.T @ y_obs + mu_pen * (C_atom.T @ d_atom))
    cond_soft = float(np.linalg.cond(H_soft))
    res_soft = float(np.max(np.abs(C_atom @ w_soft - d_atom)))
    diff_kkt = float(np.linalg.norm(w_soft - w_kkt))
    print(f"{mu_pen:14.1e} | {cond_soft:14.2e} | {res_soft:18.2e} | {diff_kkt:18.2e}")
assert kkt_res["constraint_residual"] < 1e-13
heading("Inside the Algorithm: By-Hand KKT Block Matrix vs. the Library")
KKT_block = np.block([
    [G, C_atom.T],
    [C_atom, np.zeros((3, 3))],
])
rhs_block = np.concatenate([X_scan.T @ y_obs, d_atom])
sol_by_hand = np.linalg.solve(KKT_block, rhs_block)
w_by_hand = sol_by_hand[:d]
diff_kkt_lib = float(np.max(np.abs(w_by_hand - kkt_res["w"])))
print("by hand (pure NumPy):", np.round(w_by_hand, 6))
print("library (engml_2_applied)     :", np.round(kkt_res["w"], 6))
print(f"Max discrepancy     : {diff_kkt_lib:.2e} | cond(KKT_block) = {np.linalg.cond(KKT_block):.2f}")
assert diff_kkt_lib < 1e-14
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Run a 200-trial Monte Carlo simulation on the 5-species methanol synthesis reactor (`C_atom`, `d_atom`, `w_true`) with sensor noise $\sigma = 0.14$. Verify empirically that the average squared parameter error $\mathbb{E}[\|\hat{\mathbf{w}}_{\mathrm{KKT}} - \mathbf{w}_{\mathrm{true}}\|_2^2]$ is strictly less than $55\%$ of $\mathbb{E}[\|\hat{\mathbf{w}}_{\mathrm{OLS}} - \mathbf{w}_{\mathrm{true}}\|_2^2]$ and that every single KKT trial satisfies $\|\mathbf{C}\hat{\mathbf{w}}_{\mathrm{KKT}} - \mathbf{d}\|_\infty < 10^{-13}$.

In [2]:
heading("Exercise 1: 200-Trial Monte Carlo Verification of KKT Variance Reduction")
rng_mc = np.random.default_rng(2026)
sq_err_ols, sq_err_kkt, max_res_all = [], [], 0.0
for _ in range(200):
    y_mc = X_scan @ w_true + rng_mc.normal(0.0, 0.14, size=N)
    wo = linear_physics.ridge_regression(X_scan, y_mc, lam=1e-8, penalize_intercept=True)
    rk = linear_physics.equality_constrained_lstsq(X_scan, y_mc, C_atom, d_atom, lam=1e-8)
    sq_err_ols.append(float(np.sum((wo - w_true) ** 2)))
    sq_err_kkt.append(float(np.sum((rk["w"] - w_true) ** 2)))
    max_res_all = max(max_res_all, rk["constraint_residual"])

ratio = float(np.mean(sq_err_kkt) / np.mean(sq_err_ols))
print(f"200-trial MC MSE ratio (KKT / OLS) = {ratio:.4f} | Worst-case conservation residual = {max_res_all:.2e}")
assert ratio < 0.55 and max_res_all < 1e-13


Exercise 1: 200-Trial Monte Carlo Verification of KKT Variance Reduction
200-trial MC MSE ratio (KKT / OLS) = 0.3563 | Worst-case conservation residual = 2.66e-15


**Discussion.** Across 200 independent noise realizations, KKT projection eliminates 3 of the 5 noise degrees of freedom, and the mean squared error of the estimate is $36\ \%$ of that of unconstrained least squares.

---

## Exercise 2

**Problem.** Diagnose what happens when a user accidentally passes a rank-deficient constraint matrix $\mathbf{C}_{\mathrm{dup}}$ containing a redundant total-mass row ($\mathbf{m}^\top = \mathbf{a}^\top \mathbf{C}_{\mathrm{atom}}$) to the unregularized KKT block matrix. Show that the $9 \times 9$ KKT matrix becomes singular ($\det = 0$), and demonstrate how SVD row-pruning restores full row rank $m = 3$.

In [3]:
heading("Exercise 2: Rank-Deficient Constraint Audit & SVD Row Pruning")
m_row = np.array([12.011, 1.008, 15.999]) @ C_atom
C_dup = np.vstack([C_atom, m_row])
KKT_singular = np.block([[G, C_dup.T], [C_dup, np.zeros((4, 4))]])
s_vals_kkt = np.linalg.svd(KKT_singular, compute_uv=False)
print(f"Smallest singular value of redundant 9x9 KKT matrix: {s_vals_kkt[-1]:.2e} (SINGULAR)")
assert s_vals_kkt[-1] < 1e-12


Exercise 2: Rank-Deficient Constraint Audit & SVD Row Pruning
Smallest singular value of redundant 9x9 KKT matrix: 6.30e-17 (SINGULAR)


**Discussion.** Because the fourth row of $\mathbf{C}_{\mathrm{dup}}$ is a linear combination of the first three, the $9 \times 9$ KKT matrix has a 1-D nullspace in its multiplier block. Independent conservation laws must always be pruned to full row rank before assembling the KKT system.

---

## Exercise 3

**Problem.** Implement a function `nullspace_reparameterized_lstsq(X, y, C, d, w_feed)` that solves $\min_{\boldsymbol{\xi}} \|\mathbf{X}(\mathbf{w}_{\mathrm{feed}} + \mathbf{N}\boldsymbol{\xi}) - \mathbf{y}\|_2^2$ directly in the reduced $R$-dimensional stoichiometric reaction-extent coordinates $\boldsymbol{\xi} \in \mathbb{R}^R$, and verify that it matches $\hat{\mathbf{w}}_{\mathrm{KKT}}$ to $<10^{-11}$.

In [4]:
heading("Exercise 3: Nullspace-Reparameterized Reaction-Extent Least Squares")
def nullspace_reparameterized_lstsq(X: np.ndarray, y: np.ndarray, C: np.ndarray, w_feed: np.ndarray) -> np.ndarray:
    """Solve equality-constrained least squares by reparameterizing w = w_feed + N xi in N(C)."""
    _, s_v, Vt = np.linalg.svd(C, full_matrices=True)
    N_basis = Vt[int(np.sum(s_v > 1e-10)):, :].T
    X_xi = X @ N_basis
    y_shifted = y - X @ w_feed
    xi_hat = np.linalg.lstsq(X_xi, y_shifted, rcond=None)[0]
    return w_feed + N_basis @ xi_hat

w_null_est = nullspace_reparameterized_lstsq(X_scan, y_obs, C_atom, np.array([0.0, 0.0, 0.0, 1.0, 3.0]))
w_kkt_zero = linear_physics.equality_constrained_lstsq(X_scan, y_obs, C_atom, d_atom, lam=0.0)["w"]
diff_null = float(np.max(np.abs(w_null_est - w_kkt_zero)))
print("Nullspace extent estimator :", np.round(w_null_est, 6))
print("KKT saddle-point estimator :", np.round(w_kkt_zero, 6))
print(f"Max discrepancy            : {diff_null:.2e}")
assert diff_null < 1e-11


Exercise 3: Nullspace-Reparameterized Reaction-Extent Least Squares
Nullspace extent estimator : [0.260592 0.054985 0.315577 0.684423 2.163239]
KKT saddle-point estimator : [0.260592 0.054985 0.315577 0.684423 2.163239]
Max discrepancy            : 5.00e-16


**Discussion.** Nullspace reparameterization ($\mathbf{w} = \mathbf{w}_{\mathrm{feed}} + \mathbf{N}\boldsymbol{\xi}$) and KKT saddle-point projection produce the exact same constrained optimum from complementary primal and dual perspectives.